# 01 — Inspecting and Profiling a New Dataset

Companion to [`../../data_cleaning/README.md`](../../data_cleaning/README.md).

This notebook walks through the first thing you should do with any new dataset:
look at it. Shape, dtypes, summary stats, missingness, sample rows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(0)
n = 1000
df = pd.DataFrame({
    "user_id":   rng.integers(1000, 9999, n),
    "age":       rng.normal(38, 12, n).clip(0, 100).round(),
    "income":    rng.lognormal(10.5, 0.7, n).round(2),
    "country":   rng.choice(["US", "UK", "DE", "FR", "BR", "IN"], n, p=[.35,.15,.15,.1,.1,.15]),
    "signup_dt": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 700, n), unit="D"),
    "spend":     rng.exponential(120, n).round(2),
})
# inject some realistic dirt
df.loc[rng.choice(n, 50, replace=False), "income"] = np.nan
df.loc[rng.choice(n, 5,  replace=False), "age"]    = -1            # sentinel
df.loc[rng.choice(n, 10, replace=False), "country"] = " usa "
df.loc[rng.choice(n, 3,  replace=False), "spend"]  = 1e6           # outliers
df.head()

## 1. Shape, dtypes, memory

In [ ]:
print(df.shape)
df.info(memory_usage='deep')

## 2. Per-column summary

In [ ]:
df.describe(include='all').T

## 3. Missingness

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
df.isna().astype(int).T.iloc[:, :200].pipe(lambda d: ax.imshow(d, aspect='auto', cmap='gray_r'))
ax.set_yticks(range(len(df.columns)), df.columns)
ax.set_xlabel('row (first 200)'); ax.set_title('Missingness matrix')
plt.show()

## 4. Sentinel-value hunt

In [ ]:
# Negative age?
(df['age'] < 0).sum(), df[df['age'] < 0].head()

## 5. Free-text category inspection

In [ ]:
df['country'].value_counts()  # notice ' usa ' variants

## 6. Date range and gaps

In [ ]:
df['signup_dt'].agg(['min', 'max', 'nunique'])

## What you'd do next

- Fix sentinel `-1` for `age` → `NaN`.
- Normalize country variants (`' usa '` → `'US'`).
- Investigate the spend outliers.
- See [`02_missing_values.ipynb`](02_missing_values.ipynb) and [`03_outliers.ipynb`](03_outliers.ipynb).